In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, desc, avg

# Membuat SparkSession baru
spark = SparkSession.builder \
    .appName("Tugas4_PySpark") \
    .master("local[*]") \
    .getOrCreate()

In [11]:
# A. Membaca dan Eksplorasi Awal 
# 1. Membaca dataset langsung dari HDFS
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

# 2. Tampilkan printSchema()
print("=== SKEMA DATASET ===")
df.printSchema()

# 3. Tampilkan jumlah baris (count())
print(f"Jumlah total baris: {df.count()}\n")

# 4. Tampilkan 10 baris pertama (show(10))
print("=== 10 BARIS PERTAMA ===")
df.show(10)

=== SKEMA DATASET ===
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris: 1000

=== 10 BARIS PERTAMA ===
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-0

In [12]:
# B. Menangani Data Kosong
# 1. Tampilkan jumlah baris dengan rating kosong
null_rating_count = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {null_rating_count}")

# 2. Menangani nilai kosong (Mengisi nilai kosong dengan 0 agar data transaksi penting lainnya tidak hilang)
df_clean = df.na.fill({"rating": 0})


Jumlah baris dengan rating kosong: 204


saya memilih metode df.na.fill() dengan mengisi nilai kosong menggunakan angka 0, agar baris transaksi tersebut tidak terhapus (jika memakai drop). Data transaksi tersebut masih sangat berharga untuk analisis kolom lainnya seperti total_pendapatan pada bagian berikutnya.

In [13]:
# C. Transformasi Data
# 1. Tambahkan kolom total_pendapatan (unit_terjual * harga_satuan)
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Tambahkan kolom tier_transaksi ("Besar" jika total_pendapatan > 500000, "Kecil" jika sebaliknya)
df_transformed = df_transformed.withColumn(
    "tier_transaksi", 
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

# Tampilkan beberapa hasil untuk verifikasi
df_transformed.select("order_id", "total_pendapatan", "tier_transaksi").show(5)


+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
+--------+----------------+--------------+
only showing top 5 rows



In [15]:
# D. Analisis dengan GroupBy 
# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
print("=== KATEGORI DENGAN TOTAL PENDAPATAN TERTINGGI ===")
df_transformed.groupBy("kategori") \
    .sum("total_pendapatan") \
    .withColumnRenamed("sum(total_pendapatan)", "total_pendapatan_kategori") \
    .orderBy(desc("total_pendapatan_kategori")) \
    .show(1)

# 2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
print("=== KOTA DENGAN TRANSAKSI TIER 'BESAR' TERBANYAK ===")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(desc("count")) \
    .show(1)

# 3. Berapa rata-rata rating untuk masing-masing metode_pembayaran?
print("=== RATA-RATA RATING PER METODE PEMBAYARAN ===")
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .show()

=== KATEGORI DENGAN TOTAL PENDAPATAN TERTINGGI ===
+------------+-------------------------+
|    kategori|total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|                138665000|
+------------+-------------------------+
only showing top 1 row

=== KOTA DENGAN TRANSAKSI TIER 'BESAR' TERBANYAK ===
+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

=== RATA-RATA RATING PER METODE PEMBAYARAN ===
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [17]:
# E. Menyimpan Hasil ke HDFS
# 1. Menyimpan DataFrame hasil olahan ke HDFS dalam format CSV baru
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_hasil_olahan"
df_transformed.write.mode("overwrite").option("header", "true").csv(output_path)

# 2. Verifikasi apakah penyimpanan berhasil dengan membaca kembali data tersebut
df_verify = spark.read.csv(output_path, header=True, inferSchema=True)
print(f"Verifikasi Berhasil! Jumlah baris data tersimpan: {df_verify.count()}")
df_verify.show(5)

Verifikasi Berhasil! Jumlah baris data tersimpan: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|       